In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ==============================
# HOUSE PRICES KAGGLE COMPETITION
# COMPLETE NOTEBOOK
# ==============================

# Import Libraries
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

# ==============================
# LOAD DATA
# ==============================

train_path = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv"
test_path = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

# ==============================
# PREPARE DATA
# ==============================

X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]

# Separate numerical and categorical columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

# ==============================
# PREPROCESSING
# ==============================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# ==============================
# MODEL
# ==============================

model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

# ==============================
# PIPELINE
# ==============================

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# ==============================
# TRAIN MODEL
# ==============================

pipeline.fit(X, y)

print("Model Training Complete")

# ==============================
# PREDICTIONS
# ==============================

predictions = pipeline.predict(test)

# ==============================
# CREATE SUBMISSION FILE
# ==============================

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")

# Preview
print(submission.head())